# Notebook 01: Environment Setup and Model Loading

## Goal
Verify the environment is correctly set up, load a transformer model via `transformer_lens`,
and load a matching pre-trained Sparse Autoencoder. Confirm the model and SAE are
producing expected output shapes.

## Background
We use:
- **GPT-2 Small** (default): A 124M parameter language model. 12 transformer layers,
  `d_model=768`, ~50K vocab. If sufficient resources are available, the notebook
  automatically upgrades to **Gemma 2 2B** (2.6B params, 26 layers, `d_model=2304`).
- **Pre-trained SAEs**: Sparse Autoencoders that decompose residual stream activations
  into interpretable features. GPT-2 uses Joseph Bloom's `gpt2-small-res-jb` (24,576
  features per layer); Gemma 2 uses Gemma Scope (16,384 features per layer).
- **transformer_lens**: A library that wraps transformer models with standardized
  hook names and activation caching -- critical for mechanistic interpretability.
- **sae_lens**: A library for loading and running pre-trained SAEs on hooked models.

In [3]:
import subprocess, sys, shutil, tempfile, os

print(f"Kernel Python: {sys.executable}")

if shutil.which("nvidia-smi"):
    print("NVIDIA GPU detected - installing PyTorch with CUDA 12.1...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--force-reinstall",
        "torch", "--index-url", "https://download.pytorch.org/whl/cu121", "-q"
    ])
    # Get installed torch version so we can pin it during requirements install
    r = subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__)"],
                       capture_output=True, text=True)
    torch_ver = r.stdout.strip()
    # Write a constraints file that pins torch to the CUDA version we just installed
    constraints = os.path.join(tempfile.gettempdir(), "torch_constraint.txt")
    with open(constraints, "w") as f:
        f.write(f"torch=={torch_ver}")
    # Install requirements with the constraint (pip will not change torch)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "-r", "requirements.txt", "-c", constraints, "-q"
    ])
else:
    print("No NVIDIA GPU - installing CPU-only PyTorch...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "-q"
    ])

# Verify torch build in a subprocess (fresh import)
result = subprocess.run(
    [sys.executable, "-c",
     "import torch; print(f'torch {torch.__version__}, CUDA={torch.cuda.is_available()}')"],
    capture_output=True, text=True
)
print(result.stdout.strip())
print()
print("All dependencies installed.")
print("RESTART THE KERNEL NOW, then run from Cell 2.")


Kernel Python: C:\Users\mukho\AppData\Local\Programs\Python\Python311\python.exe
NVIDIA GPU detected - installing PyTorch with CUDA 12.1...
torch 2.5.1+cu121, CUDA=True

All dependencies installed.
RESTART THE KERNEL NOW, then run from Cell 2.


In [1]:
import importlib

required = [
    'torch', 'transformer_lens', 'sae_lens', 'transformers',
    'numpy', 'pandas', 'sklearn', 'umap', 'plotly', 'matplotlib', 'tqdm'
]

missing = []
for pkg in required:
    try:
        importlib.import_module(pkg)
        print(f'{pkg}')
    except ImportError:
        missing.append(pkg)
        print(f'{pkg} ” MISSING')

if missing:
    print(f'\nInstall missing: pip install {" ".join(missing)}')
else:
    print('\nAll dependencies satisfied.')

torch
transformer_lens
sae_lens
transformers
numpy
pandas
sklearn
umap
plotly
matplotlib
tqdm

All dependencies satisfied.


In [2]:
import sys
import torch
import numpy as np

print(f"Python: {sys.executable}")
print(f"Torch:  {torch.__version__}")

# Patch: transformers >= 4.49 removed model classes from top-level.
# transformer_lens imports them but never uses them for GPT-2.
import transformers
for _cls in ("BertForPreTraining", "T5ForConditionalGeneration"):
    if not hasattr(transformers, _cls):
        setattr(transformers, _cls, type(_cls, (), {}))

from transformer_lens import HookedTransformer
from sae_lens import SAE

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"GPU:    {torch.cuda.get_device_name(0)}")
    print(f"VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    DEVICE = "cpu"
    print("CUDA not available. Using CPU.")

print(f"Device: {DEVICE}")


Python: C:\Users\mukho\AppData\Local\Programs\Python\Python311\python.exe
Torch:  2.5.1+cu121
GPU:    NVIDIA GeForce RTX 3050 Ti Laptop GPU
VRAM:   4.3 GB
Device: cuda


In [3]:
# HuggingFace Authentication
# Only strictly needed for gated models (Gemma). GPT-2 works without it.
from huggingface_hub import whoami

try:
    info = whoami()
    print("HuggingFace: logged in as", info["name"])
except Exception as e:
    print(f"HuggingFace: not authenticated ({type(e).__name__})")
    print("This is fine for GPT-2. For Gemma, run: huggingface-cli login")


HuggingFace: not authenticated (HTTPError)
This is fine for GPT-2. For Gemma, run: huggingface-cli login


## Load Gemma 2 2B

We use `transformer_lens` to load the model. Key flags:
- `fold_ln=False`: Keep LayerNorm parameters unfused â€” required for SAE compatibility
- `center_writing_weights=False`: Do not subtract mean from weight matrices
- `dtype='bfloat16'`: Half precision for memory efficiency (~5GB vs ~10GB)

**First run** will download ~5GB of weights. Subsequent runs use the HuggingFace cache.

In [4]:
# Model selection based on available resources
import torch
from transformer_lens import HookedTransformer

if torch.cuda.is_available():
    MODEL_DEVICE = "cuda"
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    MODEL_NAME = "google/gemma-2-2b" if vram_gb >= 6.0 else "gpt2"
else:
    MODEL_DEVICE = "cpu"
    MODEL_NAME = "gpt2"

MODEL_DTYPE = "bfloat16"

if MODEL_NAME == "gpt2":
    print("Using GPT-2 Small (same analysis pipeline, much smaller model)")

print(f"Loading {MODEL_NAME} on {MODEL_DEVICE} ({MODEL_DTYPE})...")

model = HookedTransformer.from_pretrained(
    MODEL_NAME,
    dtype=MODEL_DTYPE,
    fold_ln=False,
    center_writing_weights=False,
    center_unembed=False,
    device=MODEL_DEVICE,
)
model.eval()

print(f"Model loaded: {MODEL_NAME}")
print(f"  Device:  {MODEL_DEVICE}")
print(f"  Layers:  {model.cfg.n_layers}")
print(f"  d_model: {model.cfg.d_model}")
print(f"  n_heads: {model.cfg.n_heads}")


Using GPT-2 Small (same analysis pipeline, much smaller model)
Loading gpt2 on cuda (bfloat16)...


`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2 into HookedTransformer
Model loaded: gpt2
  Device:  cuda
  Layers:  12
  d_model: 768
  n_heads: 12


In [5]:
test_prompt = 'The capital of France is'
tokens = model.to_tokens(test_prompt)

with torch.no_grad():
    logits = model(tokens)

# Top 5 predicted next tokens
top5 = logits[0, -1].topk(5)
print('Top 5 predicted next tokens:')
for i, (val, idx) in enumerate(zip(top5.values, top5.indices)):
    print(f'  {i+1}. "{model.to_string([idx.item()])}" (logit={val.item():.2f})')

Top 5 predicted next tokens:
  1. " now" (logit=-103.00)
  2. " the" (logit=-103.00)
  3. " under" (logit=-103.50)
  4. " in" (logit=-103.50)
  5. " a" (logit=-103.50)


## Load a Gemma Scope SAE

Gemma Scope provides SAEs for every layer. We'll analyze layers 6, 12, and 20 
(early, middle, late). For now, load layer 12 (the semantic 'core').

SAE architecture:
- **Encoder**: `W_enc` [d_model, n_features] â€” projects activations to feature space
- **Decoder**: `W_dec` [n_features, d_model] â€” reconstructs from features
- **Bias**: `b_enc`, `b_dec`
- **Activation**: ReLU (features are non-negative)

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# 5. Load SAE — picks the right release for whichever model loaded
#
# Gemma 2 2B  → Gemma Scope (google/gemma-scope)
# GPT-2 Small → Joseph Bloom's SAEs (gpt2-small-res-jb), widely used in
#               mechanistic interpretability research
# ─────────────────────────────────────────────────────────────────────────────
from sae_lens import SAE

# Layer to analyze — scaled to model depth
#   Gemma 2 2B: 26 layers → use layer 12 (middle)
#   GPT-2 Small: 12 layers → use layer 6 (middle)
if 'gemma' in MODEL_NAME:
    TARGET_LAYER  = 12
    SAE_RELEASE   = 'gemma-scope-2b-pt-res'
    SAE_ID        = f'layer_{TARGET_LAYER}/width_16k/average_l0_71'
    TARGET_LAYERS = [6, 12, 20]   # early / middle / late
else:
    TARGET_LAYER  = 6
    SAE_RELEASE   = 'gpt2-small-res-jb'
    SAE_ID        = f'blocks.{TARGET_LAYER}.hook_resid_pre'
    TARGET_LAYERS = [2, 6, 10]    # early / middle / late (GPT-2 has 12 layers)

print(f'Loading SAE: {SAE_RELEASE}  layer {TARGET_LAYER}')

sae = SAE.from_pretrained(
    release=SAE_RELEASE,
    sae_id=SAE_ID,
    device=MODEL_DEVICE,
)
sae.eval()

print('SAE loaded.')
print(f'  Release:           {SAE_RELEASE}')
print(f'  Layer:             {TARGET_LAYER}')
print(f'  d_in (d_model):    {sae.cfg.d_in}')
print(f'  n_features:        {sae.cfg.d_sae}')
print(f'  W_enc shape:       {sae.W_enc.shape}')

Loading SAE: gpt2-small-res-jb  layer 6
SAE loaded.
  Release:           gpt2-small-res-jb
  Layer:             6
  d_in (d_model):    768
  n_features:        24576
  W_enc shape:       torch.Size([768, 24576])


C:\Users\mukho\AppData\Local\Programs\Python\Python311\Lib\site-packages\sae_lens\saes\sae.py:249: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)
  warnings.warn(


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# 7. Save config for other notebooks
# ─────────────────────────────────────────────────────────────────────────────
import json, os

config = {
    'model_name':    MODEL_NAME,
    'n_layers':      model.cfg.n_layers,
    'd_model':       model.cfg.d_model,
    'n_heads':       model.cfg.n_heads,
    'target_layers': TARGET_LAYERS,
    'sae_release':   SAE_RELEASE,
    'sae_width':     sae.cfg.d_sae,
    'model_device':  MODEL_DEVICE,
    'model_dtype':   MODEL_DTYPE,
    'seed':          SEED,
}

os.makedirs('results', exist_ok=True)
with open('results/config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('Saved to results/config.json')
print(json.dumps(config, indent=2))

Saved to results/config.json
{
  "model_name": "gpt2",
  "n_layers": 12,
  "d_model": 768,
  "n_heads": 12,
  "target_layers": [
    2,
    6,
    10
  ],
  "sae_release": "gpt2-small-res-jb",
  "sae_width": 24576,
  "model_device": "cuda",
  "model_dtype": "bfloat16",
  "seed": 42
}
